In [ ]:
# Mount Google Drive after a Colab runtime reset.
# Check the saved model and original data before restoring local files.

from pathlib import Path
import json
import torch
from google.colab import drive

drive.mount("/content/drive")

project_drive = Path("/content/drive/MyDrive/FacialVisualProfile")
run_dir = project_drive / "runs/r18_seed0"
data_source = project_drive / "data"

required_paths = [
    run_dir / "best.json",
    run_dir / "epoch_03.pt",
    run_dir / "config.json",
    run_dir / "source",
    data_source / "img_align_celeba.zip",
    data_source / "list_attr_celeba.txt",
    data_source / "list_landmarks_align_celeba.txt",
    data_source / "list_eval_partition.txt",
]

missing = []
for path in required_paths:
    exists = path.exists()
    print(f"{'FOUND' if exists else 'MISSING'}: {path.relative_to(project_drive)}")
    if not exists:
        missing.append(str(path))

assert not missing, "Some required files are missing; check the paths above."

best = json.loads(
    (run_dir / "best.json").read_text(encoding="utf-8")
)
print("\nSelected checkpoint:", best["checkpoint"])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Drive check passed.")

In [ ]:
# Restore the exact source snapshot used by the seed-0 training run.
# Extract only official validation images into a temporary Colab directory.
# Verify source hashes before importing model or dataset code.

import hashlib
import json
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0"
)
data_source = Path(
    "/content/drive/MyDrive/FacialVisualProfile/data"
)
work_dir = Path("/content/r18_day7")
work_dir.mkdir(parents=True, exist_ok=True)

config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)

for name, expected_hash in config["source_sha256"].items():
    source = run_dir / "source" / name
    contents = source.read_bytes()
    actual_hash = hashlib.sha256(contents).hexdigest()
    assert actual_hash == expected_hash, f"Source mismatch: {name}"

    destination = work_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(contents)

# Create package markers in the temporary evaluation workspace.
for package in ("models", "datasets", "evaluation"):
    (work_dir / package / "__init__.py").touch()

raw = work_dir / "data/raw"
raw.mkdir(parents=True, exist_ok=True)

for name in (
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
):
    shutil.copy2(data_source / name, raw / name)

with (raw / "list_eval_partition.txt").open() as f:
    valid_names = [
        name
        for line in f
        for name, split in [line.split()]
        if split == "1"
    ]
assert len(valid_names) == 19867

image_dir = raw / "img_align_celeba"
missing_names = [
    name for name in valid_names
    if not (image_dir / name).is_file()
]

started = time.perf_counter()
if missing_names:
    archive_path = work_dir / "img_align_celeba.zip"
    source_archive = data_source / "img_align_celeba.zip"

    if (
        not archive_path.is_file()
        or archive_path.stat().st_size != source_archive.stat().st_size
    ):
        print("Copying image archive...", flush=True)
        shutil.copy2(source_archive, archive_path)

    print(
        f"Extracting {len(missing_names)} validation images...",
        flush=True,
    )
    with zipfile.ZipFile(archive_path) as archive:
        for name in missing_names:
            archive.extract(f"img_align_celeba/{name}", path=raw)

assert all((image_dir / name).is_file() for name in valid_names)

os.chdir(work_dir)
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

print("Source hash checks passed.")
print("Validation images ready:", len(valid_names))
print("Working directory:", Path.cwd())
print(f"Preparation time: {(time.perf_counter() - started) / 60:.1f} minutes")

In [ ]:
# Load the selected checkpoint without downloading pretrained weights.
# Verify the attribute order and run a small validation forward check.
# This cell does not train or modify the checkpoint.

from pathlib import Path

import torch
from torch.utils.data import DataLoader

from datasets.celeba import CelebAAttributes
from models.resnet18 import ResNet18Attributes

assert torch.cuda.is_available(), "A GPU runtime is required."
device = torch.device("cuda")

run_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0"
)
checkpoint_path = run_dir / "epoch_03.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert checkpoint["epoch"] == 3
training_config = checkpoint["config"]

valid_dataset = CelebAAttributes(
    split="valid",
    root="/content/r18_day7/data/raw",
)

assert len(valid_dataset) == 19867
assert (
    list(valid_dataset.attribute_names)
    == list(training_config["attribute_names"])
), "Attribute order differs from the training configuration."

model = ResNet18Attributes(
    num_attributes=len(valid_dataset.attribute_names),
    pretrained=False,
)
model.load_state_dict(checkpoint["model_state"], strict=True)
model = model.to(device)
model.eval()

# Use a small batch to check checkpoint loading before full evaluation.
check_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

images, targets, points, names = next(iter(check_loader))

with torch.inference_mode():
    logits = model(images.to(device))

assert logits.shape == targets.shape == (4, 24)
assert torch.isfinite(logits).all()

print("GPU:", torch.cuda.get_device_name(0))
print("Loaded checkpoint:", checkpoint_path.name)
print("Completed epoch:", checkpoint["epoch"])
print("Validation images:", len(valid_dataset))
print("Attribute order: matched")
print("Sample filenames:", list(names))
print("Logits:", tuple(logits.shape))
print("Checkpoint loading check passed.")

In [ ]:
# Evaluate the selected checkpoint on the full official validation split.
# Save filenames, logits, and labels for later threshold selection.
# Compare the reproduced metrics with the original epoch-3 results.

import hashlib
import json
import time

import numpy as np
from torch import nn
from tqdm.auto import tqdm

from evaluation.attributes import evaluate_attributes

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

criterion = nn.BCEWithLogitsLoss()
all_logits = []
all_targets = []
all_names = []
loss_sum = 0.0
samples_seen = 0

model.eval()
started = time.perf_counter()

with torch.inference_mode():
    for images, targets, _, names in tqdm(
        valid_loader,
        desc="Evaluating validation split",
    ):
        images = images.to(device, non_blocking=True)
        gpu_targets = targets.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, gpu_targets)

        assert torch.isfinite(logits).all()
        assert torch.isfinite(loss)

        batch_size = len(names)
        loss_sum += loss.item() * batch_size
        samples_seen += batch_size

        all_logits.append(logits.cpu().numpy())
        all_targets.append(targets.numpy())
        all_names.extend(names)

elapsed_seconds = time.perf_counter() - started

valid_logits = np.concatenate(all_logits, axis=0)
valid_targets = np.concatenate(all_targets, axis=0)
valid_loss = loss_sum / samples_seen

assert samples_seen == 19867
assert valid_logits.shape == valid_targets.shape == (19867, 24)
assert all_names == valid_dataset.names

metrics = evaluate_attributes(
    valid_logits,
    valid_targets,
    threshold=0.5,
)

reference = next(
    record
    for record in checkpoint["history"]
    if record["epoch"] == 3
)

differences = {
    "valid_loss": abs(valid_loss - reference["valid_loss"]),
    "macro_f1": abs(
        metrics["macro_f1"] - reference["validation"]["macro_f1"]
    ),
    "map": abs(
        metrics["map"] - reference["validation"]["map"]
    ),
}

tolerance = 1e-4
reproduced = all(
    difference <= tolerance
    for difference in differences.values()
)

# Record the identity of the exact checkpoint used for these predictions.
checkpoint_hash = hashlib.sha256()
with checkpoint_path.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        checkpoint_hash.update(chunk)

output_dir = run_dir / "day7"
output_dir.mkdir(parents=True, exist_ok=True)

prediction_path = output_dir / "validation_predictions.npz"
temporary_path = output_dir / "validation_predictions.npz.tmp"

with temporary_path.open("wb") as handle:
    np.savez_compressed(
        handle,
        filenames=np.asarray(all_names),
        logits=valid_logits,
        targets=valid_targets,
        attribute_names=np.asarray(valid_dataset.attribute_names),
    )
temporary_path.replace(prediction_path)

report = {
    "checkpoint": checkpoint_path.name,
    "checkpoint_sha256": checkpoint_hash.hexdigest(),
    "source_sha256": training_config["source_sha256"],
    "split": "valid",
    "samples": samples_seen,
    "attribute_names": list(valid_dataset.attribute_names),
    "valid_loss": valid_loss,
    "metrics": metrics,
    "absolute_differences_from_training": differences,
    "comparison_tolerance": tolerance,
    "reproduction_passed": reproduced,
    "elapsed_seconds": elapsed_seconds,
    "environment": {
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    },
}

report_path = output_dir / "validation_reproduction.json"
temporary_report = output_dir / "validation_reproduction.json.tmp"
temporary_report.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)
temporary_report.replace(report_path)

print("Samples evaluated:", samples_seen)
print(f"Validation BCE loss: {valid_loss:.6f}")
print(f"Macro F1 at threshold 0.5: {metrics['macro_f1']:.6f}")
print(f"mAP: {metrics['map']:.6f}")
print("Absolute differences:", differences)
print("Reproduction:", "PASS" if reproduced else "NEEDS REVIEW")
print(f"Evaluation time: {elapsed_seconds / 60:.1f} minutes")
print("Saved predictions:", prediction_path)
print("Saved report:", report_path)

**Important: forget about the 0.5 threshold**:

 we now can work on a getting a better threshold by comparing threshold 0.05 - 0.95

In [ ]:
# Select one probability threshold per attribute using validation F1.
# Search thresholds from 0.05 to 0.95 in steps of 0.01.
# Break ties by choosing the threshold closest to 0.5, then the lower one.
# Save thresholds and per-attribute results without running the model again.

import csv
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import average_precision_score, f1_score

output_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0/day7"
)

reproduction = json.loads(
    (output_dir / "validation_reproduction.json").read_text(
        encoding="utf-8"
    )
)
assert reproduction["reproduction_passed"]

with np.load(
    output_dir / "validation_predictions.npz",
    allow_pickle=False,
) as cached:
    logits = cached["logits"].astype(np.float64)
    targets = cached["targets"].astype(np.int64)
    attribute_names = cached["attribute_names"].tolist()

assert logits.shape == targets.shape == (19867, 24)
assert attribute_names == reproduction["attribute_names"]
assert np.isfinite(logits).all()
assert np.isin(targets, [0, 1]).all()

# Compute sigmoid probabilities in a numerically stable way.
probabilities = np.exp(-np.logaddexp(0.0, -logits))
grid = np.arange(5, 96, dtype=np.float64) / 100.0

baseline_f1 = f1_score(
    targets,
    probabilities >= 0.5,
    average=None,
    zero_division=0,
)

f1_by_threshold = []
for threshold in grid:
    predictions = probabilities >= threshold
    true_positives = (predictions & (targets == 1)).sum(axis=0)
    false_positives = (predictions & (targets == 0)).sum(axis=0)
    false_negatives = (~predictions & (targets == 1)).sum(axis=0)

    denominator = (
        2 * true_positives + false_positives + false_negatives
    )
    scores = np.divide(
        2 * true_positives,
        denominator,
        out=np.zeros(24, dtype=np.float64),
        where=denominator != 0,
    )
    f1_by_threshold.append(scores)

f1_by_threshold = np.asarray(f1_by_threshold)
thresholds = []

for column in range(24):
    scores = f1_by_threshold[:, column]
    tied_indices = np.flatnonzero(
        np.isclose(scores, scores.max(), rtol=0, atol=1e-12)
    )
    selected_index = min(
        tied_indices,
        key=lambda index: (
            round(abs(float(grid[index]) - 0.5), 12),
            float(grid[index]),
        ),
    )
    thresholds.append(float(grid[selected_index]))

thresholds = np.asarray(thresholds)
tuned_f1 = f1_score(
    targets,
    probabilities >= thresholds,
    average=None,
    zero_division=0,
)
per_attribute_ap = average_precision_score(
    targets,
    probabilities,
    average=None,
)

assert np.all(tuned_f1 >= baseline_f1 - 1e-12)
assert np.isclose(
    baseline_f1.mean(),
    reproduction["metrics"]["macro_f1"],
    rtol=0,
    atol=1e-12,
)
assert np.isclose(
    per_attribute_ap.mean(),
    reproduction["metrics"]["map"],
    rtol=0,
    atol=1e-12,
)

threshold_report = {
    "checkpoint": reproduction["checkpoint"],
    "checkpoint_sha256": reproduction["checkpoint_sha256"],
    "fit_split": "valid",
    "samples": len(targets),
    "attribute_names": attribute_names,
    "thresholds": thresholds.tolist(),
    "objective": "per_attribute_F1",
    "grid": grid.tolist(),
    "tie_break": "closest_to_0.5_then_lower",
    "baseline_macro_f1": float(baseline_f1.mean()),
    "tuned_validation_macro_f1": float(tuned_f1.mean()),
    "map": float(per_attribute_ap.mean()),
    "note": "Thresholds were fitted and scored on the same validation split.",
}

threshold_path = output_dir / "thresholds.json"
temporary_path = output_dir / "thresholds.json.tmp"
temporary_path.write_text(
    json.dumps(threshold_report, indent=2) + "\n",
    encoding="utf-8",
)
temporary_path.replace(threshold_path)

csv_path = output_dir / "attribute_thresholds.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "attribute", "positive_count", "threshold",
        "f1_at_0.5", "tuned_validation_f1", "ap",
    ])
    for column, name in enumerate(attribute_names):
        writer.writerow([
            name,
            int(targets[:, column].sum()),
            thresholds[column],
            baseline_f1[column],
            tuned_f1[column],
            per_attribute_ap[column],
        ])

print(f"Macro F1 at 0.5: {baseline_f1.mean():.6f}")
print(f"Tuned validation macro F1: {tuned_f1.mean():.6f}")
print(f"mAP unchanged: {per_attribute_ap.mean():.6f}")
print()
print(f"{'Attribute':24s} {'Threshold':>9s} {'F1@0.5':>8s} {'Tuned F1':>9s}")
for column, name in enumerate(attribute_names):
    print(
        f"{name:24s} {thresholds[column]:9.2f} "
        f"{baseline_f1[column]:8.4f} {tuned_f1[column]:9.4f}"
    )

print("\nSaved:", threshold_path)
print("Saved:", csv_path)

now we get all the FP and FN

In [ ]:
# Summarize validation errors using the saved per-attribute thresholds.
# Inspect the four attributes with the lowest tuned validation F1.
# Randomly sample up to two false positives and two false negatives each.
# Save error statistics, selected sample details, and a contact sheet.

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

output_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0/day7"
)
image_dir = Path("/content/r18_day7/data/raw/img_align_celeba")

threshold_report = json.loads(
    (output_dir / "thresholds.json").read_text(encoding="utf-8")
)
reproduction = json.loads(
    (output_dir / "validation_reproduction.json").read_text(
        encoding="utf-8"
    )
)
assert (
    threshold_report["checkpoint_sha256"]
    == reproduction["checkpoint_sha256"]
)

with np.load(
    output_dir / "validation_predictions.npz",
    allow_pickle=False,
) as cached:
    filenames = cached["filenames"].tolist()
    attribute_names = cached["attribute_names"].tolist()
    logits = cached["logits"].astype(np.float64)
    targets = cached["targets"].astype(bool)

assert attribute_names == threshold_report["attribute_names"]

thresholds = np.asarray(threshold_report["thresholds"])
probabilities = np.exp(-np.logaddexp(0.0, -logits))
predictions = probabilities >= thresholds

tp = (predictions & targets).sum(axis=0)
fp = (predictions & ~targets).sum(axis=0)
fn = (~predictions & targets).sum(axis=0)
tn = (~predictions & ~targets).sum(axis=0)

def safe_divide(numerator, denominator):
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=np.float64),
        where=denominator != 0,
    )

precision = safe_divide(tp, tp + fp)
recall = safe_divide(tp, tp + fn)
f1 = safe_divide(2 * tp, 2 * tp + fp + fn)

statistics_path = output_dir / "validation_error_statistics.csv"
with statistics_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "attribute", "threshold", "tp", "fp", "fn", "tn",
        "precision", "recall", "f1",
    ])
    for column, name in enumerate(attribute_names):
        writer.writerow([
            name, thresholds[column],
            int(tp[column]), int(fp[column]),
            int(fn[column]), int(tn[column]),
            precision[column], recall[column], f1[column],
        ])

weakest_columns = np.argsort(f1, kind="stable")[:4]
rng = np.random.default_rng(7)
selected_samples = []

fig, axes = plt.subplots(4, 4, figsize=(14, 14))

for row, column in enumerate(weakest_columns):
    name = attribute_names[column]
    selections = []

    for error_type, mask in (
        ("FP", predictions[:, column] & ~targets[:, column]),
        ("FN", ~predictions[:, column] & targets[:, column]),
    ):
        candidates = np.flatnonzero(mask)
        chosen = rng.choice(
            candidates,
            size=min(2, len(candidates)),
            replace=False,
        )
        selections.extend(
            (error_type, int(index)) for index in chosen
        )

    for axis in axes[row]:
        axis.axis("off")

    for slot, (error_type, index) in enumerate(selections):
        filename = filenames[index]
        with Image.open(image_dir / filename) as image:
            axes[row, slot].imshow(image.convert("RGB"))

        probability = float(probabilities[index, column])
        axes[row, slot].set_title(
            f"{name} | {error_type}\n"
            f"{filename}\n"
            f"GT={int(targets[index, column])} "
            f"p={probability:.3f} t={thresholds[column]:.2f}",
            fontsize=10,
        )
        selected_samples.append({
            "attribute": name,
            "filename": filename,
            "error_type": error_type,
            "target": int(targets[index, column]),
            "probability": probability,
            "threshold": float(thresholds[column]),
        })

fig.suptitle(
    "Validation errors: random FP/FN samples for four lowest-F1 attributes",
    fontsize=14,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))

figure_path = output_dir / "validation_error_examples.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

sample_path = output_dir / "validation_error_examples.json"
sample_path.write_text(
    json.dumps({
        "checkpoint_sha256": threshold_report["checkpoint_sha256"],
        "split": "valid",
        "sampling_seed": 7,
        "selection_rule": "four_lowest_tuned_F1_attributes",
        "samples": selected_samples,
    }, indent=2) + "\n",
    encoding="utf-8",
)

print(f"{'Attribute':24s} {'Precision':>9s} {'Recall':>8s} {'FP':>6s} {'FN':>6s}")
for column in weakest_columns:
    print(
        f"{attribute_names[column]:24s} "
        f"{precision[column]:9.4f} {recall[column]:8.4f} "
        f"{fp[column]:6d} {fn[column]:6d}"
    )

print("Saved:", statistics_path)
print("Saved:", sample_path)
print("Saved:", figure_path)

In [ ]:
# Package the Day 7 thresholds, small reports, and English summary.
# Preserve project-relative paths for later extraction on the Mac.
# Keep model checkpoints, prediction arrays, and images in Google Drive.

import json
import zipfile
from pathlib import Path

from google.colab import files

day7_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0/day7"
)

threshold_report = json.loads(
    (day7_dir / "thresholds.json").read_text(encoding="utf-8")
)
reproduction = json.loads(
    (day7_dir / "validation_reproduction.json").read_text(
        encoding="utf-8"
    )
)

assert reproduction["reproduction_passed"]
assert (
    threshold_report["checkpoint_sha256"]
    == reproduction["checkpoint_sha256"]
)

report_names = (
    "validation_reproduction.json",
    "attribute_thresholds.csv",
    "validation_error_statistics.csv",
    "validation_error_examples.json",
)

for name in ("thresholds.json", *report_names):
    path = day7_dir / name
    assert path.is_file() and path.stat().st_size > 0, name

summary = f"""# Day 7: ResNet18 validation and threshold selection

- Model: ResNet18, seed 0, selected checkpoint `epoch_03.pt`.
- Validation samples: 19,867, using the official CelebA split.
- Training source files were verified against their saved SHA-256 hashes.
- Checkpoint loading and attribute-order checks passed.
- Validation BCE loss: {reproduction["valid_loss"]:.6f}.
- Macro F1 at threshold 0.5: {threshold_report["baseline_macro_f1"]:.6f}.
- Validation mAP: {threshold_report["map"]:.6f}.
- Reproduced BCE loss, macro F1, and mAP matched the original epoch-3 records exactly.
- Selected one threshold per attribute by maximizing validation F1.
- Search grid: 0.05 to 0.95 inclusive, in steps of 0.01.
- Tie rule: closest to 0.5, then the lower threshold.
- Tuned validation macro F1: {threshold_report["tuned_validation_macro_f1"]:.6f}.
- Thresholds were fitted and scored on the same validation split; this is not a held-out estimate of improvement.
- Model weights and mAP were unchanged by threshold selection.
- Computed per-attribute TP, FP, FN, TN, precision, recall, and F1.
- Sampled FP/FN examples for the four lowest tuned-F1 attributes using seed 7.
- Two Bags_Under_Eyes examples raised visual interpretation questions; annotation errors were not confirmed.
- Bags_Under_Eyes refers to eye bags, not dark circles.
- Official labels were retained.
- The test split was not evaluated.
- Full prediction arrays and the error-example figure remain in Drive under `FacialVisualProfile/runs/r18_seed0/day7/`.
"""

summary_path = day7_dir / "day7_r18_validation.md"
summary_path.write_text(summary, encoding="utf-8")

archive_path = Path("/content/day7_results.zip")

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        day7_dir / "thresholds.json",
        arcname="configs/thresholds/r18_seed0.json",
    )
    archive.write(
        summary_path,
        arcname="docs/day7_r18_validation.md",
    )
    for name in report_names:
        archive.write(
            day7_dir / name,
            arcname=f"docs/results/r18_seed0/{name}",
        )

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    print("Packaged files:")
    for name in archive.namelist():
        print(" ", name)

print("Saved summary:", summary_path)
print("Download:", archive_path.name)
files.download(str(archive_path))